# 02. Data Preprocessing

**Objectives:**
- Load raw banking data
- Clean and preprocess OHLCV data
- Add technical indicators
- Prepare data for agent analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('default')
sns.set_palette('husl')

In [ ]:
# Load the raw data
data_path = '../data/banking_ohlcv_raw.csv'
df = pd.read_csv(data_path, header=[0, 1], index_col=0)
df.index = pd.to_datetime(df.index)

print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Date range:", df.index.min(), "to", df.index.max())
df.head()

In [ ]:
# Clean data - remove any NaN values
df_clean = df.dropna()
print(f"Original shape: {df.shape}, Cleaned shape: {df_clean.shape}")

# Check for any remaining issues
print("Missing values per column:")
print(df_clean.isnull().sum())

In [ ]:
def add_technical_indicators(df, ticker):
    """Add technical indicators for a single ticker"""
    data = df[ticker].copy()
    
    # Simple Moving Averages
    data['SMA_20'] = data['Close'].rolling(window=20).mean()
    data['SMA_50'] = data['Close'].rolling(window=50).mean()
    
    # Exponential Moving Averages
    data['EMA_12'] = data['Close'].ewm(span=12).mean()
    data['EMA_26'] = data['Close'].ewm(span=26).mean()
    
    # MACD
    data['MACD'] = data['EMA_12'] - data['EMA_26']
    data['Signal_Line'] = data['MACD'].ewm(span=9).mean()
    
    # RSI
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))
    
    # Bollinger Bands
    data['BB_Middle'] = data['Close'].rolling(window=20).mean()
    data['BB_Upper'] = data['BB_Middle'] + 2 * data['Close'].rolling(window=20).std()
    data['BB_Lower'] = data['BB_Middle'] - 2 * data['Close'].rolling(window=20).std()
    
    # Daily returns
    data['Daily_Return'] = data['Close'].pct_change()
    
    # Volatility (20-day rolling std of returns)
    data['Volatility'] = data['Daily_Return'].rolling(window=20).std()
    
    return data

# Apply to all tickers
tickers = ['COMB.N0000.LK', 'HNB.N0000.LK', 'NDB.N0000.LK', 'BFLN.N0000.LK', 'DFCC.N0000.LK', 'SAMP.N0000.LK']
processed_data = {}

for ticker in tickers:
    if ticker in df_clean.columns.levels[0]:
        processed_data[ticker] = add_technical_indicators(df_clean, ticker)
        print(f"Processed {ticker}")
    else:
        print(f"Ticker {ticker} not found in data")

# Combine back
df_processed = pd.concat(processed_data, axis=1, keys=tickers)
print(f"Processed data shape: {df_processed.shape}")

In [ ]:
# Visualize one ticker's data
ticker = 'COMB.N0000.LK'
if ticker in processed_data:
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    
    # Price chart with moving averages
    axes[0].plot(processed_data[ticker]['Close'], label='Close Price', alpha=0.7)
    axes[0].plot(processed_data[ticker]['SMA_20'], label='SMA 20', alpha=0.7)
    axes[0].plot(processed_data[ticker]['SMA_50'], label='SMA 50', alpha=0.7)
    axes[0].fill_between(processed_data[ticker].index, 
                        processed_data[ticker]['BB_Lower'], 
                        processed_data[ticker]['BB_Upper'], 
                        alpha=0.1, color='blue', label='Bollinger Bands')
    axes[0].set_title(f'{ticker} Price Analysis')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # RSI
    axes[1].plot(processed_data[ticker]['RSI'], color='purple')
    axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.7)
    axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.7)
    axes[1].set_title('RSI (Relative Strength Index)')
    axes[1].set_ylim(0, 100)
    axes[1].grid(True, alpha=0.3)
    
    # MACD
    axes[2].plot(processed_data[ticker]['MACD'], label='MACD', color='blue')
    axes[2].plot(processed_data[ticker]['Signal_Line'], label='Signal Line', color='red')
    axes[2].bar(processed_data[ticker].index, 
               processed_data[ticker]['MACD'] - processed_data[ticker]['Signal_Line'], 
               alpha=0.3, color='gray', label='Histogram')
    axes[2].set_title('MACD')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Ticker {ticker} not available for visualization")

In [ ]:
# Save processed data
output_path = '../data/banking_processed.csv'
df_processed.to_csv(output_path)
print(f"Processed data saved to {output_path}")

# Summary statistics
print("\nSummary of processed data:")
for ticker in tickers:
    if ticker in processed_data:
        print(f"\n{ticker}:")
        print(f"  Records: {len(processed_data[ticker])}")
        print(f"  Avg Close: {processed_data[ticker]['Close'].mean():.2f}")
        print(f"  Volatility: {processed_data[ticker]['Volatility'].mean():.4f}")